# Picoclimate Unsupervised Windowed Feature Pipeline

This notebook builds an end-to-end, unsupervised pipeline for windowed feature extraction, clustering, and explainability using track files under `data/picoclimate_test/cities`.

Data layout:
- `cities/<city>/<season>/<YYYY-MM-DD>/<time_slot>/<track_id>.csv`

Supervisor math for one track file:
$$
23 \text{ variables} \times 100 \text{ windows} \times 11 \text{ stats} = 25{,}300 \text{ features}
$$

If you use all track files (currently 490 files), the master matrix is:
$$
490 \text{ rows} \times 25{,}300 \text{ features}
$$

The sections below follow the full pipeline: data inspection, track loading, missing-value handling, window discovery, feature extraction, dimensionality reduction, clustering, and explainability.

## Environment and optional dependencies

Core dependencies: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

Optional packages (only used if installed):
- `umap-learn` for UMAP embeddings
- `hdbscan` for density clustering
- `shap` for explainability
- `stumpy` for matrix profile motifs

If you need them, install with:
```
pip install umap-learn hdbscan shap stumpy
```

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier

import importlib.util as importlib_util


def _has_package(name: str) -> bool:
    return importlib_util.find_spec(name) is not None


HAVE_UMAP = _has_package("umap")
HAVE_HDBSCAN = _has_package("hdbscan")
HAVE_SHAP = _has_package("shap")
HAVE_STUMPY = _has_package("stumpy")

if HAVE_UMAP:
    import umap

if HAVE_HDBSCAN:
    import hdbscan

if HAVE_SHAP:
    import shap

if HAVE_STUMPY:
    import stumpy

In [6]:
BASE_DIR = Path("data/picoclimate_test/cities")
META_PATH = Path(r"D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\metadata.json")
FIELDS_PATH = Path(r"D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\csv_fields_explained.json")
OUTPUT_DIR = Path("outputs/picoclimate_shapelets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = ["summer", "winter"]
SLOTS = ["morning", "noon", "afternoon", "night"]

WINDOW_COUNT = 100
WINDOW_LENGTHS = [10, 20, 30]
STATS = [
    "mean",
    "median",
    "min",
    "max",
    "std",
    "diff1_mean",
    "diff2_mean",
    "q10",
    "q25",
    "q75",
    "q90",
]

RNG_SEED = 20260603

# Metadata says tracks have 100..150 locations; use 150 slots (loc_000..loc_149).
MASTER_LOC_COUNT = 150

## 1. Inspect Folder Structure and Metadata

Walk `data/picoclimate_test/cities`, read `csv_fields_explained.json`, and build a dataframe of file metadata for stratified slicing.

In [7]:
with META_PATH.open("r", encoding="utf-8") as f:
    meta_json = json.load(f)

with FIELDS_PATH.open("r", encoding="utf-8") as f:
    fields_json = json.load(f)

track_files = sorted(BASE_DIR.rglob("*.csv"))
rows = []
for path in track_files:
    rel = path.relative_to(BASE_DIR)
    if len(rel.parts) != 5:
        continue
    city, season, date_str, time_slot, filename = rel.parts
    track_id = path.stem
    rows.append(
        {
            "city": city,
            "season": season,
            "date": date_str,
            "time_slot": time_slot,
            "track_id": track_id,
            "path": str(path),
        }
    )

meta_df = pd.DataFrame(rows)
meta_df["date"] = pd.to_datetime(meta_df["date"], errors="coerce")

print("Total track files:", len(meta_df))
print(meta_df.head(3))

counts = (
    meta_df.groupby(["city", "season", "time_slot"])\
    .size()\
    .reset_index(name="n_files")
    .sort_values(["city", "season", "time_slot"])
)
counts

KeyError: 'date'

## 2. Load and Standardize Track Files

Helper functions to load a track CSV, normalize column names, and return a tidy variable x loc_index matrix.

In [ ]:
LOC_RE = re.compile(r"_loc_(\d+)$")


def extract_loc_index(col: str):
    match = LOC_RE.search(col)
    return int(match.group(1)) if match else None


def build_master_loc_cols(count: int = MASTER_LOC_COUNT):
    return [f"loc_{i:03d}" for i in range(count)]


MASTER_LOC_COLS = build_master_loc_cols()


def load_track_matrix(path: Path, master_cols=MASTER_LOC_COLS):
    df = pd.read_csv(path)
    if "variable" not in df.columns:
        raise ValueError(f"Missing 'variable' column in {path}")

    df = df.set_index("variable")
    loc_cols = [c for c in df.columns if extract_loc_index(c) is not None]
    loc_idx = [extract_loc_index(c) for c in loc_cols]

    rename = {c: f"loc_{i:03d}" for c, i in zip(loc_cols, loc_idx)}
    df = df[loc_cols].rename(columns=rename)

    df = df.reindex(columns=master_cols)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


# Optional: infer maximum loc index from files (slow on large datasets).
# def infer_master_loc_cols(meta_df):
#     max_idx = -1
#     for path_str in meta_df["path"]:
#         df_head = pd.read_csv(path_str, nrows=1)
#         for col in df_head.columns:
#             idx = extract_loc_index(col)
#             if idx is not None:
#                 max_idx = max(max_idx, idx)
#     return [f"loc_{i:03d}" for i in range(max_idx + 1)]

## 3. Reindex to Master loc_000..loc_149 and Impute Missing Values

Reindex each track to the master loc index, interpolate across gaps, and then fill dead zones using the mean matrix of the same (city, season, time_slot) group.

In [ ]:
_group_mean_cache = {}


def interpolate_track(df: pd.DataFrame) -> pd.DataFrame:
    return df.interpolate(method="linear", axis=1, limit_direction="both")


def compute_group_mean(meta_df: pd.DataFrame, city: str, season: str, time_slot: str):
    key = (city, season, time_slot)
    if key in _group_mean_cache:
        return _group_mean_cache[key]

    group = meta_df[
        (meta_df["city"] == city)
        & (meta_df["season"] == season)
        & (meta_df["time_slot"] == time_slot)
    ]

    if group.empty:
        return None

    sum_mat = None
    count_mat = None
    variables = None

    for path_str in group["path"]:
        df = load_track_matrix(Path(path_str), MASTER_LOC_COLS)
        df = interpolate_track(df)
        arr = df.to_numpy()

        if sum_mat is None:
            variables = df.index
            sum_mat = np.zeros_like(arr, dtype=float)
            count_mat = np.zeros_like(arr, dtype=float)

        mask = ~np.isnan(arr)
        sum_mat += np.where(mask, arr, 0.0)
        count_mat += mask.astype(float)

    mean_mat = np.divide(
        sum_mat,
        count_mat,
        out=np.full_like(sum_mat, np.nan),
        where=count_mat > 0,
    )
    mean_df = pd.DataFrame(mean_mat, index=variables, columns=MASTER_LOC_COLS)
    _group_mean_cache[key] = mean_df
    return mean_df


def impute_dead_zones(df: pd.DataFrame, group_mean: pd.DataFrame) -> pd.DataFrame:
    if group_mean is None:
        return df
    dead_cols = df.isna().all(axis=0)
    if dead_cols.any():
        df = df.copy()
        df.loc[:, dead_cols] = group_mean.loc[:, dead_cols]
    return df


def prepare_track(path_str: str, meta_df: pd.DataFrame):
    path = Path(path_str)
    rel = path.relative_to(BASE_DIR)
    city, season, date_str, time_slot, _ = rel.parts

    df = load_track_matrix(path, MASTER_LOC_COLS)
    df = interpolate_track(df)
    group_mean = compute_group_mean(meta_df, city, season, time_slot)
    df = impute_dead_zones(df, group_mean)

    info = {
        "city": city,
        "season": season,
        "date": date_str,
        "time_slot": time_slot,
        "track_id": path.stem,
    }
    return df, info

## 4. Discover or Sample 100 Windows (Matrix Profile / Random Sampling)

Define 100 windows along the loc_index axis. Use matrix profile motifs when available, or default to random sampling with multiple window lengths.

In [ ]:
USE_MATRIX_PROFILE = False


def sample_windows(n_locs: int, window_count=WINDOW_COUNT, lengths=WINDOW_LENGTHS, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    windows = []
    for _ in range(window_count):
        length = int(rng.choice(lengths))
        if length >= n_locs:
            length = max(2, n_locs - 1)
        start = int(rng.integers(0, n_locs - length + 1))
        end = start + length
        windows.append((start, end))
    return windows


def matrix_profile_windows(series, window_lengths=WINDOW_LENGTHS, per_length=40):
    if not HAVE_STUMPY:
        raise RuntimeError("stumpy is not installed")

    series = np.asarray(series, dtype=float)
    if np.all(np.isnan(series)):
        raise ValueError("series contains only NaN values")

    series = np.nan_to_num(series, nan=np.nanmean(series))
    windows = []

    for m in window_lengths:
        if len(series) <= m + 2:
            continue
        mp = stumpy.stump(series, m)
        top_idx = np.argsort(mp[:, 0])[:per_length]
        for idx in top_idx:
            windows.append((int(idx), int(idx + m)))

    return windows[:WINDOW_COUNT]


def build_windows(meta_df: pd.DataFrame, master_cols=MASTER_LOC_COLS):
    n_locs = len(master_cols)

    if USE_MATRIX_PROFILE and HAVE_STUMPY:
        sample_path = meta_df["path"].iloc[0]
        sample_df, _ = prepare_track(sample_path, meta_df)
        sample_series = sample_df.iloc[0].to_numpy()
        windows = matrix_profile_windows(sample_series)
    else:
        windows = sample_windows(n_locs)

    return windows


windows = build_windows(meta_df, MASTER_LOC_COLS)
print("Window count:", len(windows))
print("First 5 windows:", windows[:5])

## 5. Compute 11 Statistics per Window and Flatten to 25,300 Features

Compute 11 statistics per (variable, window) and flatten into a single feature vector.

In [ ]:
def _safe_stats(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0 or np.all(np.isnan(arr)):
        return {name: np.nan for name in STATS}

    stats = {
        "mean": np.nanmean(arr),
        "median": np.nanmedian(arr),
        "min": np.nanmin(arr),
        "max": np.nanmax(arr),
        "std": np.nanstd(arr),
    }

    diff1 = np.diff(arr)
    diff2 = np.diff(arr, n=2)
    stats["diff1_mean"] = np.nanmean(diff1) if diff1.size else np.nan
    stats["diff2_mean"] = np.nanmean(diff2) if diff2.size else np.nan

    q10, q25, q75, q90 = np.nanpercentile(arr, [10, 25, 75, 90])
    stats["q10"] = q10
    stats["q25"] = q25
    stats["q75"] = q75
    stats["q90"] = q90

    return stats


def build_feature_names(variables, windows):
    names = []
    for var in variables:
        for w_idx in range(len(windows)):
            for stat in STATS:
                names.append(f"{var}__w{w_idx:03d}__{stat}")
    return names


def extract_feature_vector(df: pd.DataFrame, windows):
    features = []
    for var in df.index:
        row = df.loc[var].to_numpy()
        for start, end in windows:
            stats = _safe_stats(row[start:end])
            features.extend([stats[s] for s in STATS])
    return np.asarray(features, dtype=float)

## 6. Assemble Master Feature Matrix for a Slice

Loop through a stratified slice (city/season/time_slot), build the feature matrix, and store row and column metadata.

In [ ]:
slice_city = "Nantes"
slice_season = "summer"
slice_slot = "morning"

slice_meta = meta_df[
    (meta_df["city"] == slice_city)
    & (meta_df["season"] == slice_season)
    & (meta_df["time_slot"] == slice_slot)
].copy()

if slice_meta.empty:
    raise ValueError("Selected slice has no files")

sample_df = load_track_matrix(Path(slice_meta["path"].iloc[0]), MASTER_LOC_COLS)
variables = sample_df.index.tolist()

feature_names = build_feature_names(variables, windows)

rows = []
row_meta = []

for _, row in slice_meta.iterrows():
    df, info = prepare_track(row["path"], meta_df)
    vec = extract_feature_vector(df, windows)
    rows.append(vec)
    row_meta.append(info)

X = np.vstack(rows)
X_df = pd.DataFrame(X, columns=feature_names)
row_meta_df = pd.DataFrame(row_meta)

print("Feature matrix shape:", X_df.shape)
row_meta_df.head(3)

## 7. Scale Features and Reduce Dimensionality (PCA/UMAP)

Standardize features, then produce a low-dimensional embedding for clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

pca_components = min(20, X_scaled.shape[1])
pca = PCA(n_components=pca_components, random_state=RNG_SEED)
X_pca = pca.fit_transform(X_scaled)

embedding = X_pca
embedding_name = "pca"

if HAVE_UMAP:
    umap_model = umap.UMAP(n_components=2, random_state=RNG_SEED)
    X_umap = umap_model.fit_transform(X_scaled)
    embedding = X_umap
    embedding_name = "umap"

print("Embedding:", embedding_name, embedding.shape)

## 8. Cluster with HDBSCAN or K-Means

Cluster the embedding, tune hyperparameters, and visualize clusters.

In [ ]:
if HAVE_HDBSCAN:
    clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
    labels = clusterer.fit_predict(embedding)
else:
    kmeans = KMeans(n_clusters=6, random_state=RNG_SEED, n_init=10)
    labels = kmeans.fit_predict(embedding)

row_meta_df["cluster"] = labels

# Visualize first two dimensions
plt.figure(figsize=(6, 5))
plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap="tab20", s=15)
plt.title(f"Clusters ({embedding_name})")
plt.xlabel("dim1")
plt.ylabel("dim2")
plt.show()

## 9. Explain Clusters with Tree Models and SHAP

Train a Random Forest on the original feature space and review top feature importances. Use SHAP when available.

In [ ]:
valid_mask = labels != -1 if HAVE_HDBSCAN else np.ones_like(labels, dtype=bool)
X_train = X_scaled[valid_mask]
y_train = labels[valid_mask]

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=RNG_SEED,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

importances = pd.Series(clf.feature_importances_, index=feature_names)
importances.sort_values(ascending=False).head(20)

In [ ]:
if HAVE_SHAP:
    sample_size = min(200, X_train.shape[0])
    rng = np.random.default_rng(RNG_SEED)
    sample_idx = rng.choice(X_train.shape[0], size=sample_size, replace=False)
    sample = X_train[sample_idx]

    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(sample)

    # This renders a summary plot in notebooks with graphical output.
    shap.summary_plot(shap_values, sample, feature_names=feature_names, show=True)

## Save Outputs

Persist feature matrices, embeddings, labels, and window definitions for downstream analysis.

In [ ]:
slice_tag = f"{slice_city}_{slice_season}_{slice_slot}"

X_df.to_csv(OUTPUT_DIR / f"features_{slice_tag}.csv", index=False)
row_meta_df.to_csv(OUTPUT_DIR / f"rows_{slice_tag}.csv", index=False)

np.save(OUTPUT_DIR / f"embedding_{embedding_name}_{slice_tag}.npy", embedding)

with (OUTPUT_DIR / f"windows_{slice_tag}.json").open("w", encoding="utf-8") as f:
    json.dump(
        [{"start": int(s), "end": int(e)} for s, e in windows],
        f,
        indent=2,
    )

print("Saved outputs to", OUTPUT_DIR)

## Optional: Run All Slices

To build features for every (city, season, time_slot) slice, loop over groups and reuse the same helpers. This can be time-consuming on large datasets.

In [ ]:
# Example loop to build per-slice feature matrices.
# This reuses the same windows to keep feature columns aligned.
#
# for (city, season, slot), group in meta_df.groupby(["city", "season", "time_slot"]):
#     rows = []
#     row_meta = []
#     for _, row in group.iterrows():
#         df, info = prepare_track(row["path"], meta_df)
#         vec = extract_feature_vector(df, windows)
#         rows.append(vec)
#         row_meta.append(info)
#
#     X = np.vstack(rows)
#     X_df = pd.DataFrame(X, columns=feature_names)
#     row_meta_df = pd.DataFrame(row_meta)
#
#     slice_tag = f"{city}_{season}_{slot}"
#     X_df.to_csv(OUTPUT_DIR / f"features_{slice_tag}.csv", index=False)
#     row_meta_df.to_csv(OUTPUT_DIR / f"rows_{slice_tag}.csv", index=False)